In [97]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 9 modules, rebound 10 names.


In [98]:
import altair as alt
import numpy as np
import polars as pl
import torch

from ts_uncertainty.forecasting import ForecastModel
from ts_uncertainty.metrics import mean_absolute_scaled_error, scaled_quantile_loss

alt.renderers.enable("jupyter", offline=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [99]:
from ts_uncertainty.chaotic import ChaoticGenerator
from ts_uncertainty.injection import Injector
from ts_uncertainty.plotting import plot_sample, plot_forecasts
from ts_uncertainty.synthetic import SyntheticGenerator
from ts_uncertainty.real import RealSampler

# data settings
context_length = 256
horizon_length = 128
samples_batch = 100
seed = 42

## Synthetic Samples

In [5]:
sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=False,
    periods=(24,),             # period of the sine
    # n_periods=1,
    n_harmonics=3,             # 1 harmonic = pure sine, no extra shape
    # level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    # amplitude_range=(1.0, 1.0),
    seed=seed
)

# generate samples
samples_synthetic = sin_gen.sample_batch(samples_batch)

In [6]:
sample_to_plot = 0
plot_sample(samples_synthetic[sample_to_plot], title=f"Synthetic Sample {sample_to_plot}")

## Chaotic Samples

In [7]:
chaos_gen = ChaoticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    systems=None,                    # None = all ~120 default systems
    dim=0,                           # observed coordinate, None = random per sample
    pts_per_period_range=(20, 30),   # time steps per dominant period (smaller = more irregular)
    level_range=(0.0, 0.0),          # centered at 0
    slope_range=(0.0, 0.0),          # no trend
    amplitude_range=(1.0, 1.0),      # z-normalised signal, std ≈ 1
    seed=seed,
)

# generate samples
samples_chaotic = chaos_gen.sample_batch(samples_batch)

In [8]:
sample_to_plot = 0
plot_sample(samples_chaotic[sample_to_plot], title=f"Chaotic Sample {sample_to_plot}")

## Real Sample
Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

In [9]:
data_path_test = "../data/test.csv"
data_test = pl.read_csv(data_path_test, try_parse_dates=True)

number_series = len(data_test['series_index'].unique())
print(f"Number of time series in the test set: {number_series}")
data_test.head()

Number of time series in the test set: 361


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2025-11-19 10:00:00,0.36,"""home_electricity""",0
2025-11-19 10:15:00,0.124,"""home_electricity""",0
2025-11-19 10:30:00,0.108,"""home_electricity""",0
2025-11-19 10:45:00,0.088,"""home_electricity""",0
2025-11-19 11:00:00,0.116,"""home_electricity""",0


In [10]:
real_gen = RealSampler(
    df=data_test,
    context_length=context_length,
    horizon=horizon_length,
    normalize='none', # dont normalize
    seed=seed,
)

# generate samples
samples_real = real_gen.sample_batch(samples_batch)

In [11]:
sample_to_plot = 0
plot_sample(samples_real[sample_to_plot], title=f"Real Sample {sample_to_plot}")

# Models

In [12]:
models = {
    "AutoTheta": {"name": "autotheta"},
    "Chronos-2": {'name': 'chronos2'},
    "TiRex-2": {'name': 'tirex2'},
    "Timestral (123M)": {'name': 'timestral', 'size': '123M'},
}

# prepare and download all models
for key, m in models.items():
    print(f"Preparing model: {key}")
    model = ForecastModel(**m)

Preparing model: AutoTheta
Preparing model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Preparing model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Preparing model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

# Experiments

In [78]:
def evaluate_models_on_samples(models, samples):
    results = []
    scores_mase = []
    scores_sql = []

    for key, m in models.items():
        print(f"Evaluating model: {key}")
        model = ForecastModel(**m)
        fc = model.forecast(samples)

        fc_m = fc.metric_inputs()
        score_mase = mean_absolute_scaled_error(fc_m["y_true"], fc_m["y_pred"], fc_m["y_train"], fc_m["season_length"])
        score_sql = scaled_quantile_loss(fc_m["y_true"], fc_m["y_quantiles"], fc_m["y_train"], fc_m["quantiles"], fc_m["season_length"])

        results.append(fc)
        scores_mase.append(score_mase.mean())
        scores_sql.append(score_sql.mean())

    scores = pl.DataFrame({
        "model": list(models.keys()),
        "mase": [float(x.squeeze()) for x in scores_mase],
        "sql": [float(x.squeeze()) for x in scores_sql],
    })
    return results, scores


def summarize_scores(scores: dict) -> pl.DataFrame:
    """Long score table, one row per (dataset, model).

    scores: either one run {dataset: DataFrame}, giving columns mase, sql, ...;
    or several runs {run: {dataset: DataFrame}}, giving columns mase_<run>, sql_<run>, ...
    """
    def to_long(fc_scores: dict[str, pl.DataFrame]) -> pl.DataFrame:
        long = pl.concat([df.with_columns(pl.lit(name).alias("dataset")) for name, df in fc_scores.items()])
        return long.select("dataset", "model", pl.exclude("dataset", "model"))

    if all(isinstance(v, pl.DataFrame) for v in scores.values()):  # single run
        return to_long(scores)

    tables = []
    for run, fc_scores in scores.items():
        long = to_long(fc_scores)
        tables.append(long.select("dataset", "model", *[pl.col(m).alias(f"{m}_{run}") for m in long.columns[2:]]))

    out = tables[0]
    for t in tables[1:]:
        out = out.join(t, on=["dataset", "model"], how="full", coalesce=True, maintain_order="left")

    # group columns by metric: mase_clean, mase_noise_h, ..., sql_clean, sql_noise_h, ...
    runs = list(scores)
    metric_cols = sorted(out.columns[2:], key=lambda c: (c.split("_", 1)[0], runs.index(c.split("_", 1)[1])))
    return out.select("dataset", "model", *metric_cols)



In [75]:
batch_size = 100  # define the batch size for evaluation
data_samples = {
    "synthetic": samples_synthetic[:batch_size],
    "chaotic": samples_chaotic[:batch_size],
    "real": samples_real[:batch_size],
}

## No Injection

In [76]:
fc_results = {}
fc_scores = {}
for type_s, s in data_samples.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results[type_s] = results
    fc_scores[type_s] = scores

Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [79]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.000 ┆ 3.014 │
│ synthetic ┆ Chronos-2        ┆ 0.055 ┆ 0.045 │
│ synthetic ┆ TiRex-2          ┆ 0.128 ┆ 0.129 │
│ synthetic ┆ Timestral (123M) ┆ 0.034 ┆ 0.028 │
│ chaotic   ┆ AutoTheta        ┆ 5.331 ┆ 4.844 │
│ chaotic   ┆ Chronos-2        ┆ 3.192 ┆ 2.639 │
│ chaotic   ┆ TiRex-2          ┆ 3.116 ┆ 2.467 │
│ chaotic   ┆ Timestral (123M) ┆ 2.926 ┆ 2.302 │
│ real      ┆ AutoTheta        ┆ 2.021 ┆ 2.349 │
│ real      ┆ Chronos-2        ┆ 1.129 ┆ 0.911 │
│ real      ┆ TiRex-2          ┆ 1.230 ┆ 0.986 │
│ real      ┆ Timestral (123M) ┆ 1.316 ┆ 0.999 │
└───────────┴──────────────────┴───────┴───────┘


In [50]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results[dataset_to_plot]))
plot_forecasts(
    data_samples[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/1661976593.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


## Noise

### Horizon only

In [100]:
inj = Injector(
    noise=True,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    noise_region="horizon",
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_noise_h = {}
fc_scores_noise_h = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_noise_h[type_s] = results
    fc_scores_noise_h[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [101]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_noise_h))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.772 ┆ 3.149 │
│ synthetic ┆ Chronos-2        ┆ 0.776 ┆ 0.737 │
│ synthetic ┆ TiRex-2          ┆ 0.790 ┆ 0.683 │
│ synthetic ┆ Timestral (123M) ┆ 0.774 ┆ 0.751 │
│ chaotic   ┆ AutoTheta        ┆ 5.382 ┆ 4.876 │
│ chaotic   ┆ Chronos-2        ┆ 3.324 ┆ 2.754 │
│ chaotic   ┆ TiRex-2          ┆ 3.231 ┆ 2.562 │
│ chaotic   ┆ Timestral (123M) ┆ 3.059 ┆ 2.418 │
│ real      ┆ AutoTheta        ┆ 2.134 ┆ 2.389 │
│ real      ┆ Chronos-2        ┆ 1.311 ┆ 1.092 │
│ real      ┆ TiRex-2          ┆ 1.399 ┆ 1.138 │
│ real      ┆ Timestral (123M) ┆ 1.500 ┆ 1.167 │
└───────────┴──────────────────┴───────┴───────┘


In [102]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_noise_h[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/2482385561.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


### Context only

In [103]:
inj = Injector(
    noise=True,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    noise_region="context",
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_noise_c = {}
fc_scores_noise_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_noise_c[type_s] = results
    fc_scores_noise_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [104]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_noise_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.286 ┆ 1.365 │
│ synthetic ┆ Chronos-2        ┆ 0.198 ┆ 0.250 │
│ synthetic ┆ TiRex-2          ┆ 0.341 ┆ 0.373 │
│ synthetic ┆ Timestral (123M) ┆ 0.220 ┆ 0.278 │
│ chaotic   ┆ AutoTheta        ┆ 3.638 ┆ 3.574 │
│ chaotic   ┆ Chronos-2        ┆ 2.137 ┆ 1.727 │
│ chaotic   ┆ TiRex-2          ┆ 2.151 ┆ 1.703 │
│ chaotic   ┆ Timestral (123M) ┆ 2.113 ┆ 1.655 │
│ real      ┆ AutoTheta        ┆ 1.334 ┆ 1.424 │
│ real      ┆ Chronos-2        ┆ 0.947 ┆ 0.756 │
│ real      ┆ TiRex-2          ┆ 0.962 ┆ 0.769 │
│ real      ┆ Timestral (123M) ┆ 0.924 ┆ 0.736 │
└───────────┴──────────────────┴───────┴───────┘


In [105]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_noise_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/4188322343.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


### Both

In [106]:
inj = Injector(
    noise=True,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    noise_region="both",
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_noise_b = {}
fc_scores_noise_b = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_noise_b[type_s] = results
    fc_scores_noise_b[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [107]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_noise_b))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.767 ┆ 1.460 │
│ synthetic ┆ Chronos-2        ┆ 0.742 ┆ 0.576 │
│ synthetic ┆ TiRex-2          ┆ 0.801 ┆ 0.631 │
│ synthetic ┆ Timestral (123M) ┆ 0.749 ┆ 0.581 │
│ chaotic   ┆ AutoTheta        ┆ 3.671 ┆ 3.591 │
│ chaotic   ┆ Chronos-2        ┆ 2.218 ┆ 1.789 │
│ chaotic   ┆ TiRex-2          ┆ 2.218 ┆ 1.753 │
│ chaotic   ┆ Timestral (123M) ┆ 2.190 ┆ 1.712 │
│ real      ┆ AutoTheta        ┆ 1.365 ┆ 1.439 │
│ real      ┆ Chronos-2        ┆ 1.011 ┆ 0.801 │
│ real      ┆ TiRex-2          ┆ 1.020 ┆ 0.808 │
│ real      ┆ Timestral (123M) ┆ 0.987 ┆ 0.777 │
└───────────┴──────────────────┴───────┴───────┘


In [112]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_noise_b[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/2017913783.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


### Summary

In [113]:
summary = summarize_scores({
    "clean": fc_scores,
    "noise_h": fc_scores_noise_h,
    "noise_c": fc_scores_noise_c,
    "noise_b": fc_scores_noise_b,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)

shape: (12, 10)
┌───────────┬──────────────────┬────────────┬──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_noise_h ┆ mase_noise_c ┆ mase_noise_b ┆ sql_clean ┆ sql_noise_h ┆ sql_noise_c ┆ sql_noise_b │
│ ---       ┆ ---              ┆ ---        ┆ ---          ┆ ---          ┆ ---          ┆ ---       ┆ ---         ┆ ---         ┆ ---         │
│ str       ┆ str              ┆ f64        ┆ f64          ┆ f64          ┆ f64          ┆ f64       ┆ f64         ┆ f64         ┆ f64         │
╞═══════════╪══════════════════╪════════════╪══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ synthetic ┆ AutoTheta        ┆ 0.000      ┆ 0.772        ┆ 0.286        ┆ 0.767        ┆ 3.014     ┆ 3.149       ┆ 1.365       ┆ 1.460       │
│ synthetic ┆ Chronos-2        ┆ 0.055      ┆ 0.776        ┆ 0.198        ┆ 0.742        ┆ 0.045     ┆ 0.737      

## Spikes

### Without covariates

#### Horizon only

In [144]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="horizon",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_h = {}
fc_scores_spike_h = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_h[type_s] = results
    fc_scores_spike_h[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [145]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_h))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.755 ┆ 3.485 │
│ synthetic ┆ Chronos-2        ┆ 0.808 ┆ 0.796 │
│ synthetic ┆ TiRex-2          ┆ 0.878 ┆ 0.873 │
│ synthetic ┆ Timestral (123M) ┆ 0.788 ┆ 0.781 │
│ chaotic   ┆ AutoTheta        ┆ 5.718 ┆ 5.136 │
│ chaotic   ┆ Chronos-2        ┆ 3.658 ┆ 3.088 │
│ chaotic   ┆ TiRex-2          ┆ 3.583 ┆ 2.907 │
│ chaotic   ┆ Timestral (123M) ┆ 3.401 ┆ 2.751 │
│ real      ┆ AutoTheta        ┆ 2.428 ┆ 2.685 │
│ real      ┆ Chronos-2        ┆ 1.580 ┆ 1.352 │
│ real      ┆ TiRex-2          ┆ 1.679 ┆ 1.419 │
│ real      ┆ Timestral (123M) ┆ 1.769 ┆ 1.438 │
└───────────┴──────────────────┴───────┴───────┘


In [146]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_h[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/2991431023.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Context only

In [147]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="context",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_c = {}
fc_scores_spike_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_c[type_s] = results
    fc_scores_spike_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [148]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.879 ┆ 1.426 │
│ synthetic ┆ Chronos-2        ┆ 0.102 ┆ 0.091 │
│ synthetic ┆ TiRex-2          ┆ 0.281 ┆ 0.260 │
│ synthetic ┆ Timestral (123M) ┆ 0.107 ┆ 0.125 │
│ chaotic   ┆ AutoTheta        ┆ 2.407 ┆ 2.725 │
│ chaotic   ┆ Chronos-2        ┆ 1.425 ┆ 1.151 │
│ chaotic   ┆ TiRex-2          ┆ 1.440 ┆ 1.136 │
│ chaotic   ┆ Timestral (123M) ┆ 1.343 ┆ 1.051 │
│ real      ┆ AutoTheta        ┆ 1.180 ┆ 1.034 │
│ real      ┆ Chronos-2        ┆ 0.809 ┆ 0.638 │
│ real      ┆ TiRex-2          ┆ 0.825 ┆ 0.641 │
│ real      ┆ Timestral (123M) ┆ 0.790 ┆ 0.617 │
└───────────┴──────────────────┴───────┴───────┘


In [149]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/1422315983.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (period)

In [150]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="both",
    spike_prob=0.04,
    spike_period=48,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_bp = {}
fc_scores_spike_bp = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_bp[type_s] = results
    fc_scores_spike_bp[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [151]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_bp))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.951 ┆ 2.291 │
│ synthetic ┆ Chronos-2        ┆ 0.161 ┆ 0.130 │
│ synthetic ┆ TiRex-2          ┆ 0.679 ┆ 0.609 │
│ synthetic ┆ Timestral (123M) ┆ 0.403 ┆ 0.335 │
│ chaotic   ┆ AutoTheta        ┆ 3.360 ┆ 3.677 │
│ chaotic   ┆ Chronos-2        ┆ 2.022 ┆ 1.653 │
│ chaotic   ┆ TiRex-2          ┆ 2.135 ┆ 1.705 │
│ chaotic   ┆ Timestral (123M) ┆ 2.004 ┆ 1.589 │
│ real      ┆ AutoTheta        ┆ 1.378 ┆ 2.128 │
│ real      ┆ Chronos-2        ┆ 0.954 ┆ 0.760 │
│ real      ┆ TiRex-2          ┆ 1.042 ┆ 0.823 │
│ real      ┆ Timestral (123M) ┆ 1.016 ┆ 0.804 │
└───────────┴──────────────────┴───────┴───────┘


In [152]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_bp[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/1268868358.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (random)

In [153]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    spike_region="both",
    spike_prob=0.04,
    spike_period=None,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_br = {}
fc_scores_spike_br = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_br[type_s] = results
    fc_scores_spike_br[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [154]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_br))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.305 ┆ 1.809 │
│ synthetic ┆ Chronos-2        ┆ 0.669 ┆ 0.655 │
│ synthetic ┆ TiRex-2          ┆ 0.836 ┆ 0.805 │
│ synthetic ┆ Timestral (123M) ┆ 0.672 ┆ 0.682 │
│ chaotic   ┆ AutoTheta        ┆ 2.442 ┆ 2.734 │
│ chaotic   ┆ Chronos-2        ┆ 1.651 ┆ 1.369 │
│ chaotic   ┆ TiRex-2          ┆ 1.674 ┆ 1.351 │
│ chaotic   ┆ Timestral (123M) ┆ 1.560 ┆ 1.262 │
│ real      ┆ AutoTheta        ┆ 1.309 ┆ 1.190 │
│ real      ┆ Chronos-2        ┆ 0.924 ┆ 0.760 │
│ real      ┆ TiRex-2          ┆ 0.931 ┆ 0.763 │
│ real      ┆ Timestral (123M) ┆ 0.925 ┆ 0.754 │
└───────────┴──────────────────┴───────┴───────┘


In [155]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_br[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/2361099547.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Summary

In [156]:
summary = summarize_scores({
    "clean": fc_scores,
    "spike_h": fc_scores_spike_h,
    "spike_c": fc_scores_spike_c,
    "spike_bp": fc_scores_spike_bp,
    "spike_br": fc_scores_spike_br,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)

shape: (12, 12)
┌───────────┬──────────────────┬────────────┬──────────────┬──────────────┬───────────────┬───────────────┬───────────┬─────────────┬─────────────┬──────────────┬──────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_spike_h ┆ mase_spike_c ┆ mase_spike_bp ┆ mase_spike_br ┆ sql_clean ┆ sql_spike_h ┆ sql_spike_c ┆ sql_spike_bp ┆ sql_spike_br │
│ ---       ┆ ---              ┆ ---        ┆ ---          ┆ ---          ┆ ---           ┆ ---           ┆ ---       ┆ ---         ┆ ---         ┆ ---          ┆ ---          │
│ str       ┆ str              ┆ f64        ┆ f64          ┆ f64          ┆ f64           ┆ f64           ┆ f64       ┆ f64         ┆ f64         ┆ f64          ┆ f64          │
╞═══════════╪══════════════════╪════════════╪══════════════╪══════════════╪═══════════════╪═══════════════╪═══════════╪═════════════╪═════════════╪══════════════╪══════════════╡
│ synthetic ┆ AutoTheta        ┆ 0.000      ┆ 0.755        ┆ 0.879        ┆ 0.951         ┆ 1.

### With covariates

#### Horizon only

In [157]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="horizon",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_h_c = {}
fc_scores_spike_h_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_h_c[type_s] = results
    fc_scores_spike_h_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [158]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_h_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.755 ┆ 3.485 │
│ synthetic ┆ Chronos-2        ┆ 0.824 ┆ 0.807 │
│ synthetic ┆ TiRex-2          ┆ 0.872 ┆ 0.870 │
│ synthetic ┆ Timestral (123M) ┆ 0.790 ┆ 0.782 │
│ chaotic   ┆ AutoTheta        ┆ 5.718 ┆ 5.136 │
│ chaotic   ┆ Chronos-2        ┆ 3.670 ┆ 3.017 │
│ chaotic   ┆ TiRex-2          ┆ 3.545 ┆ 2.871 │
│ chaotic   ┆ Timestral (123M) ┆ 3.488 ┆ 2.775 │
│ real      ┆ AutoTheta        ┆ 2.428 ┆ 2.685 │
│ real      ┆ Chronos-2        ┆ 1.763 ┆ 1.479 │
│ real      ┆ TiRex-2          ┆ 1.653 ┆ 1.405 │
│ real      ┆ Timestral (123M) ┆ 1.766 ┆ 1.441 │
└───────────┴──────────────────┴───────┴───────┘


In [159]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_h_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/115649765.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Context only

In [160]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="context",
    spike_prob=0.04,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_c_c = {}
fc_scores_spike_c_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_c_c[type_s] = results
    fc_scores_spike_c_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [161]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_c_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.879 ┆ 1.426 │
│ synthetic ┆ Chronos-2        ┆ 0.089 ┆ 0.071 │
│ synthetic ┆ TiRex-2          ┆ 0.292 ┆ 0.264 │
│ synthetic ┆ Timestral (123M) ┆ 0.124 ┆ 0.107 │
│ chaotic   ┆ AutoTheta        ┆ 2.407 ┆ 2.725 │
│ chaotic   ┆ Chronos-2        ┆ 1.403 ┆ 1.112 │
│ chaotic   ┆ TiRex-2          ┆ 1.466 ┆ 1.156 │
│ chaotic   ┆ Timestral (123M) ┆ 1.372 ┆ 1.071 │
│ real      ┆ AutoTheta        ┆ 1.180 ┆ 1.034 │
│ real      ┆ Chronos-2        ┆ 0.803 ┆ 0.626 │
│ real      ┆ TiRex-2          ┆ 0.811 ┆ 0.632 │
│ real      ┆ Timestral (123M) ┆ 0.797 ┆ 0.616 │
└───────────┴──────────────────┴───────┴───────┘


In [162]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_c_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/179687162.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (period)

In [163]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="both",
    spike_prob=0.04,
    spike_period=48,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_bp_c = {}
fc_scores_spike_bp_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_bp_c[type_s] = results
    fc_scores_spike_bp_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta


Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [164]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_bp_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.951 ┆ 2.291 │
│ synthetic ┆ Chronos-2        ┆ 0.165 ┆ 0.133 │
│ synthetic ┆ TiRex-2          ┆ 0.509 ┆ 0.440 │
│ synthetic ┆ Timestral (123M) ┆ 0.324 ┆ 0.261 │
│ chaotic   ┆ AutoTheta        ┆ 3.360 ┆ 3.677 │
│ chaotic   ┆ Chronos-2        ┆ 1.967 ┆ 1.568 │
│ chaotic   ┆ TiRex-2          ┆ 2.039 ┆ 1.629 │
│ chaotic   ┆ Timestral (123M) ┆ 1.927 ┆ 1.516 │
│ real      ┆ AutoTheta        ┆ 1.378 ┆ 2.128 │
│ real      ┆ Chronos-2        ┆ 0.983 ┆ 0.784 │
│ real      ┆ TiRex-2          ┆ 0.970 ┆ 0.761 │
│ real      ┆ Timestral (123M) ┆ 0.940 ┆ 0.735 │
└───────────┴──────────────────┴───────┴───────┘


In [165]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_bp_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/436871585.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Both (random)

In [166]:
inj = Injector(
    noise=False,
    spikes=True,
    shift=False,
    shift_covariate=False,
    spike_covariate=True,
    spike_region="both",
    spike_prob=0.04,
    spike_period=None,
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_spike_br_c = {}
fc_scores_spike_br_c = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_spike_br_c[type_s] = results
    fc_scores_spike_br_c[type_s] = scores

Injecting anomalies into synthetic samples..., number of samples: 100
Injecting anomalies into chaotic samples..., number of samples: 100
Injecting anomalies into real samples..., number of samples: 100
Forecasting synthetic samples..., number of samples: 100
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [167]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores_spike_br_c))

shape: (12, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 1.305 ┆ 1.809 │
│ synthetic ┆ Chronos-2        ┆ 0.772 ┆ 0.703 │
│ synthetic ┆ TiRex-2          ┆ 0.903 ┆ 0.819 │
│ synthetic ┆ Timestral (123M) ┆ 0.740 ┆ 0.621 │
│ chaotic   ┆ AutoTheta        ┆ 2.442 ┆ 2.734 │
│ chaotic   ┆ Chronos-2        ┆ 1.696 ┆ 1.348 │
│ chaotic   ┆ TiRex-2          ┆ 1.691 ┆ 1.364 │
│ chaotic   ┆ Timestral (123M) ┆ 1.580 ┆ 1.252 │
│ real      ┆ AutoTheta        ┆ 1.309 ┆ 1.190 │
│ real      ┆ Chronos-2        ┆ 0.935 ┆ 0.756 │
│ real      ┆ TiRex-2          ┆ 0.930 ┆ 0.753 │
│ real      ┆ Timestral (123M) ┆ 0.913 ┆ 0.721 │
└───────────┴──────────────────┴───────┴───────┘


In [168]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 5

forecasts = dict(zip(models.keys(), fc_results_spike_br_c[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_1369/2868376069.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


#### Summary

In [169]:
summary = summarize_scores({
    "clean": fc_scores,
    "spike_h_c": fc_scores_spike_h_c,
    "spike_c_c": fc_scores_spike_c_c,
    "spike_bp_c": fc_scores_spike_bp_c,
    "spike_br_c": fc_scores_spike_br_c,
})
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3, tbl_width_chars=1000):
    print(summary)

shape: (12, 12)
┌───────────┬──────────────────┬────────────┬────────────────┬────────────────┬─────────────────┬─────────────────┬───────────┬───────────────┬───────────────┬────────────────┬────────────────┐
│ dataset   ┆ model            ┆ mase_clean ┆ mase_spike_h_c ┆ mase_spike_c_c ┆ mase_spike_bp_c ┆ mase_spike_br_c ┆ sql_clean ┆ sql_spike_h_c ┆ sql_spike_c_c ┆ sql_spike_bp_c ┆ sql_spike_br_c │
│ ---       ┆ ---              ┆ ---        ┆ ---            ┆ ---            ┆ ---             ┆ ---             ┆ ---       ┆ ---           ┆ ---           ┆ ---            ┆ ---            │
│ str       ┆ str              ┆ f64        ┆ f64            ┆ f64            ┆ f64             ┆ f64             ┆ f64       ┆ f64           ┆ f64           ┆ f64            ┆ f64            │
╞═══════════╪══════════════════╪════════════╪════════════════╪════════════════╪═════════════════╪═════════════════╪═══════════╪═══════════════╪═══════════════╪════════════════╪════════════════╡
│ synthetic ┆ 

## Shift

### Without covariates

#### Horizon only

In [ ]:
inj = Injector(
    noise=True,
    spikes=False,
    shift=False,
    shift_covariate=False,
    spike_covariate=False,
    noise_region="horizon",
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real") for x in s]

fc_results_noise_h = {}
fc_scores_noise_h = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results_noise_h[type_s] = results
    fc_scores_noise_h[type_s] = scores

#### Context only

#### Both

#### Summary

#### Special, check all shift points

### With covariates

#### Horizon only

#### Context only

#### Both

#### Summary

#### Special, check all shift points